In [ ]:
# =====================================================================
# BOLUM 1 - Ayarlar, klasor yapisi ve veri yukleme
# =====================================================================
import os
import numpy as np
import pandas as pd

RANDOM_STATE = 42   # tekrarlanabilirlik (bu asamada kullanilmiyor, standart olarak tanimli)

# ---- Portable project paths ---------------------------------------------
from pathlib import Path

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    required = (
        "masonry_tower_primary_dataset.xlsx",
        "masonry_tower_geometry_reference_dataset.xlsx",
    )
    for candidate in (current, *current.parents):
        data_dir = candidate / "data"
        if all((data_dir / name).is_file() for name in required):
            return candidate
    raise FileNotFoundError(
        "Project root could not be located. Run this notebook from the repository "
        "root or from its code/ directory, and keep the data/ directory unchanged."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GEO_FILE = DATA_DIR / "masonry_tower_geometry_reference_dataset.xlsx"
MAT_FILE = DATA_DIR / "masonry_tower_primary_dataset.xlsx"
# ------------------------------------------------------------------------

# Sutun tanimlari (dosyalardaki adlarla birebir ayni olmalidir)
GEO_COLS = ["Height (m)", "Section a (m)", "Section b (m)", "Wall Thickness (m)",
            "Opening z/H", "Opening Ratio x (%)", "Opening Ratio y (%)"]
MAT_COLS = ["E (MPa)", "d (kg/m3)"]
TARGETS  = ["f1 (Hz)", "f2 (Hz)"]

# Geometri dosyasinin sabit malzeme degerleri (esleme kontrolu icin)
E_REF = 3000
D_REF = 1400

# Konsol ciktisini hem ekrana yaz hem dosyaya kaydet
LOG_LINES = []
def log(msg=""):
    print(msg)
    LOG_LINES.append(str(msg))

# Sayfa adlari iki dosyada farkli oldugu icin indeks ile okunuyor
df_geo = pd.read_excel(GEO_FILE, sheet_name=0)
df_mat = pd.read_excel(MAT_FILE, sheet_name=0)

log("Veri dosyalari yuklendi.")
log(f"Geometry dataset  : {df_geo.shape[0]} rows x {df_geo.shape[1]} columns")
log(f"Material dataset  : {df_mat.shape[0]} rows x {df_mat.shape[1]} columns")

In [ ]:
# =====================================================================
# BOLUM 2 - Veri yapisi ve veri kalitesi kontrolu
# =====================================================================

def veri_yapisi_raporu(df, dataset_name, expected_cols):
    """Bir veri kumesinin yapisini ve kalitesini kontrol eder, ozet tablo dondurur."""
    log("\n" + "=" * 70)
    log(f"DATASET: {dataset_name}")
    log("=" * 70)

    # Boyut ve sutun adlari
    log(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    log("Columns: " + ", ".join(map(str, df.columns)))

    # Beklenen sutunlar mevcut mu
    eksik_sutun = [c for c in expected_cols if c not in df.columns]
    fazla_sutun = [c for c in df.columns if c not in expected_cols]
    log(f"Missing expected columns : {eksik_sutun if eksik_sutun else 'None'}")
    log(f"Unexpected extra columns : {fazla_sutun if fazla_sutun else 'None'}")

    # Veri tipleri ve ilk bes satir
    log("\nData types:")
    log(df.dtypes.to_string())
    log("\nFirst 5 rows:")
    log(df.head(5).to_string())

    # Eksik deger ve tam tekrarli satir kontrolu
    log(f"\nTotal missing values : {int(df.isna().sum().sum())}")
    log(f"Fully duplicated rows: {int(df.duplicated().sum())}")

    # Sayisala cevrilemeyen (hatali) kayit kontrolu
    ozet = []
    for c in df.columns:
        seri_num = pd.to_numeric(df[c], errors="coerce")
        ozet.append({
            "Variable": c,
            "Dtype": str(df[c].dtype),
            "Missing": int(df[c].isna().sum()),
            "Non_numeric_entries": int(seri_num.isna().sum() - df[c].isna().sum()),
            "Unique_values": int(df[c].nunique()),
            "Min": seri_num.min(),
            "Max": seri_num.max(),
            "Zero_count": int((seri_num == 0).sum()),
            "Negative_count": int((seri_num < 0).sum()),
        })
    ozet_df = pd.DataFrame(ozet)
    log("\nColumn-wise quality summary:")
    log(ozet_df.to_string(index=False))
    return ozet_df


ozet_geo = veri_yapisi_raporu(df_geo, "GEOMETRY-ONLY DATASET", GEO_COLS + TARGETS)
ozet_mat = veri_yapisi_raporu(df_mat, "MATERIAL-INCLUDED DATASET", GEO_COLS + MAT_COLS + TARGETS)

# --- Fiziksel tutarlilik kontrolleri (yalnizca raporlama, silme yok) ---
log("\n" + "=" * 70)
log("PHYSICAL CONSISTENCY CHECKS (reporting only, no rows removed)")
log("=" * 70)

for ad, d in [("Geometry-only", df_geo), ("Material-included", df_mat)]:
    f2_kucuk = int((d["f2 (Hz)"] < d["f1 (Hz)"]).sum())    # f2 normalde f1'den buyuk beklenir
    f_sifir   = int(((d["f1 (Hz)"] <= 0) | (d["f2 (Hz)"] <= 0)).sum())
    oran_hata = int(((d["Opening z/H"] < 0) | (d["Opening z/H"] > 1)).sum())
    log(f"{ad}: rows with f2 < f1 = {f2_kucuk} | rows with non-positive f1 or f2 = {f_sifir} "
        f"| rows with Opening z/H outside [0,1] = {oran_hata}")

# Kalite ozet tablolarinin disa aktarilmasi
with pd.ExcelWriter(os.path.join(TABLE_DIR, "01_data_structure_summary.xlsx"),
                    engine="openpyxl") as writer:
    ozet_geo.to_excel(writer, sheet_name="Geometry_dataset", index=False)
    ozet_mat.to_excel(writer, sheet_name="Material_dataset", index=False)

In [ ]:
# =====================================================================
# BOLUM 3 - Temel tanimlayici istatistikler
# =====================================================================

def tanimlayici_istatistik(df, dataset_name):
    """count, mean, std, min, Q1, median, Q3, max, skewness, kurtosis tablosu uretir."""
    sayisal = df.select_dtypes(include=[np.number])
    tablo = pd.DataFrame({
        "Variable": sayisal.columns,
        "Count": sayisal.count().values,
        "Mean": sayisal.mean().values,
        "Std": sayisal.std().values,
        "Min": sayisal.min().values,
        "Q1": sayisal.quantile(0.25).values,
        "Median": sayisal.median().values,
        "Q3": sayisal.quantile(0.75).values,
        "Max": sayisal.max().values,
        "Skewness": sayisal.skew().values,
        "Kurtosis": sayisal.kurtosis().values,   # Fisher tanimi (normal dagilim = 0)
    })
    # Makale tablosu icin okunabilir yuvarlama
    tablo.iloc[:, 1:] = tablo.iloc[:, 1:].astype(float).round(4)

    log("\n" + "=" * 70)
    log(f"DESCRIPTIVE STATISTICS - {dataset_name}")
    log("=" * 70)
    log(tablo.to_string(index=False))
    return tablo


ist_geo = tanimlayici_istatistik(df_geo, "GEOMETRY-ONLY DATASET")
ist_mat = tanimlayici_istatistik(df_mat, "MATERIAL-INCLUDED DATASET")

ist_geo.to_excel(os.path.join(TABLE_DIR, "02_descriptive_statistics_geometry.xlsx"), index=False)
ist_mat.to_excel(os.path.join(TABLE_DIR, "03_descriptive_statistics_material.xlsx"), index=False)

In [ ]:
# =====================================================================
# BOLUM 4 - Geometry_ID olusturulmasi ve grup yapisinin dogrulanmasi
# =====================================================================

ROUND_DEC = 6   # float karsilastirma hatalarini onlemek icin yuvarlama basamagi

def geometri_anahtari(df, cols=GEO_COLS, ndec=ROUND_DEC):
    """Yedi geometrik parametreden benzersiz bir metin anahtar uretir."""
    return df[cols].round(ndec).astype(str).agg("|".join, axis=1)

df_mat["Geo_Key"] = geometri_anahtari(df_mat)
df_geo["Geo_Key"] = geometri_anahtari(df_geo)

# Malzemeli dosyadaki gorulme sirasina gore G001, G002, ... etiketleri
benzersiz_anahtarlar = pd.unique(df_mat["Geo_Key"])
anahtar_to_id = {k: f"G{i+1:03d}" for i, k in enumerate(benzersiz_anahtarlar)}
df_mat["Geometry_ID"] = df_mat["Geo_Key"].map(anahtar_to_id)
df_geo["Geometry_ID"] = df_geo["Geo_Key"].map(anahtar_to_id)   # eslesmeyenler NaN kalir

log("\n" + "=" * 70)
log("GEOMETRY GROUP VALIDATION (material-included dataset)")
log("=" * 70)
log(f"Total rows                        : {len(df_mat)}")
log(f"Number of unique geometries       : {df_mat['Geometry_ID'].nunique()}")

# Her geometriye ait kayit sayisi ve benzersiz malzeme durumu sayisi
grup = df_mat.groupby("Geometry_ID").agg(
    Record_count=("Geometry_ID", "size"),
    Unique_material_cases=("E (MPa)", lambda s: len(set(zip(s, df_mat.loc[s.index, "d (kg/m3)"]))))
).reset_index()

log("\nDistribution of records per geometry:")
log(grup["Record_count"].value_counts().sort_index().to_string())

# Alti kayittan farkli olan geometriler
sorunlu = grup[(grup["Record_count"] != 6) | (grup["Unique_material_cases"] != 6)]
log(f"\nGeometries NOT having exactly 6 records / 6 unique material cases: {len(sorunlu)}")
if len(sorunlu) > 0:
    log(sorunlu.to_string(index=False))

# Ayni geometri + ayni malzeme kombinasyonunun tekrari
tekrar_mask = df_mat.duplicated(subset=["Geometry_ID"] + MAT_COLS, keep=False)
log(f"Rows with duplicated (Geometry_ID, E, d) combination: {int(tekrar_mask.sum())}")

with pd.ExcelWriter(os.path.join(TABLE_DIR, "04_geometry_group_check.xlsx"),
                    engine="openpyxl") as writer:
    grup.to_excel(writer, sheet_name="Records_per_geometry", index=False)
    sorunlu.to_excel(writer, sheet_name="Irregular_geometries", index=False)
    df_mat.loc[tekrar_mask].to_excel(writer, sheet_name="Duplicated_material_cases", index=False)

# Geometry_ID eklenmis veri kumesi sonraki asamalarda kullanilacak
df_mat.drop(columns=["Geo_Key"]).to_excel(
    os.path.join(TABLE_DIR, "material_dataset_with_geometry_id.xlsx"), index=False)

In [ ]:
# =====================================================================
# BOLUM 5 - Geometri dosyasi ile malzemeli dosyanin esleme kontrolu
# =====================================================================

log("\n" + "=" * 70)
log("CROSS-FILE CONSISTENCY CHECK")
log("=" * 70)

geo_anahtar = set(df_geo["Geo_Key"])
mat_anahtar = set(df_mat["Geo_Key"])
log(f"Unique geometries in geometry file : {len(geo_anahtar)}")
log(f"Unique geometries in material file : {len(mat_anahtar)}")
log(f"Common geometries                  : {len(geo_anahtar & mat_anahtar)}")
log(f"Only in geometry file              : {len(geo_anahtar - mat_anahtar)}")
log(f"Only in material file              : {len(mat_anahtar - geo_anahtar)}")

sadece_geo = df_geo[df_geo["Geo_Key"].isin(geo_anahtar - mat_anahtar)]
sadece_mat = df_mat[df_mat["Geo_Key"].isin(mat_anahtar - geo_anahtar)]

# Referans malzeme durumu (E = 3000 MPa, d = 1400 kg/m3) satirlari
ref = df_mat[(df_mat["E (MPa)"] == E_REF) & (df_mat["d (kg/m3)"] == D_REF)].copy()
log(f"\nRows in material file with E={E_REF} MPa and d={D_REF} kg/m3 : {len(ref)}")

# f1 ve f2 degerlerinin karsilastirilmasi
karsilastirma = df_geo.merge(
    ref[["Geo_Key"] + TARGETS], on="Geo_Key", how="left", suffixes=("_geo", "_mat"))

TOL = 1e-4   # mutlak fark toleransi (Hz)
karsilastirma["f1_abs_diff"] = (karsilastirma["f1 (Hz)_geo"] - karsilastirma["f1 (Hz)_mat"]).abs()
karsilastirma["f2_abs_diff"] = (karsilastirma["f2 (Hz)_geo"] - karsilastirma["f2 (Hz)_mat"]).abs()
karsilastirma["Match"] = (karsilastirma["f1_abs_diff"] <= TOL) & (karsilastirma["f2_abs_diff"] <= TOL)

log(f"Matched geometries (|diff| <= {TOL} Hz) : {int(karsilastirma['Match'].sum())}")
log(f"Mismatched or unmatched geometries      : {int((~karsilastirma['Match']).sum())}")
log(f"Max |f1 difference| : {karsilastirma['f1_abs_diff'].max()}")
log(f"Max |f2 difference| : {karsilastirma['f2_abs_diff'].max()}")

eslesmeyen = karsilastirma[~karsilastirma["Match"]]

with pd.ExcelWriter(os.path.join(TABLE_DIR, "05_cross_file_matching_check.xlsx"),
                    engine="openpyxl") as writer:
    karsilastirma.to_excel(writer, sheet_name="Full_comparison", index=False)
    eslesmeyen.to_excel(writer, sheet_name="Mismatched_records", index=False)
    sadece_geo.to_excel(writer, sheet_name="Only_in_geometry_file", index=False)
    sadece_mat.to_excel(writer, sheet_name="Only_in_material_file", index=False)

# Konsol ciktisinin kaydedilmesi
with open(os.path.join(TABLE_DIR, "00_console_log.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(LOG_LINES))

log("\nTum kontroller tamamlandi. Cikti klasoru: " + TABLE_DIR)